# Exercises: Linear Regression

**Practical Machine Learning** · Sayan CHAKI · LIRIS, Université Lyon 2

Three exercises:

| # | Topic | Main skills |
|---|---|---|
| 1 | **Gradient descent from scratch** on house energy data | normal equations, learning rate, why scaling speeds up GD, back to original units |
| 2 | **Nucleus area from radius** (breast cancer images) | residual plots, feature engineering, a model with physical meaning |
| 3 | **Predicting the alcohol content of wines** | baselines, Ridge, k-NN regression, coefficient stability under the bootstrap |

**Open in Colab.** In [colab.research.google.com](https://colab.research.google.com) choose *File → Upload notebook* (or open it from Google Drive). Everything uses datasets shipped with scikit-learn, so no download or upload of data is needed.

**Rules of the game**
* Keep all `random_state` / seeds as given, so results are comparable across the class.
* Never use the test set to choose a model or a hyperparameter; it is opened **once**, at the end.
* Cells marked `# TODO` are yours. Cells marked `# CHECK` test your work: run them, they must pass.
* Questions marked ✍️ need a short written answer (2 to 4 sentences, with numbers from your results).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
# Exercise 1 · Gradient descent from scratch

**Context (simulated data).** Daily energy consumption (kWh) of 300 houses, explained by the number of occupants, the outdoor temperature (°C) and the floor area (m²). The three features live on very different scales. Because the data are simulated, the true coefficients are known.

In [ ]:
rng = np.random.default_rng(0)
n = 300
occupants = rng.integers(1, 7, n).astype(float)
temperature = rng.uniform(-5, 25, n)
area = rng.uniform(30, 250, n)
X = np.column_stack([occupants, temperature, area])
y = 2 + 1.5 * occupants - 0.3 * temperature + 0.05 * area + rng.normal(0, 1, n)
features = ["occupants", "temperature", "area"]
print(pd.DataFrame(X, columns=features).describe().loc[["mean", "std", "min", "max"]].round(2))
print("true parameters: b = 2, w = [1.5, -0.3, 0.05]")

### 1.1 The exact solution
Write `fit_normal_equations(X, y)`: add a column of ones in front of `X`, then solve $X^\top X\,\theta = X^\top y$ with `np.linalg.solve` (never invert the matrix explicitly). Return $\theta=(b, w_1, \dots, w_p)$.

In [ ]:
# TODO
def add_ones(X):
    return np.column_stack([np.ones(len(X)), X])

def fit_normal_equations(X, y):
    raise NotImplementedError("TODO")

In [ ]:
# CHECK (run this cell, it must pass)
theta_ne = fit_normal_equations(X, y)
ref = LinearRegression().fit(X, y)
assert np.allclose(theta_ne, np.r_[ref.intercept_, ref.coef_])
print("✅ normal equations:", np.round(theta_ne, 4))
mse_opt = np.mean((y - add_ones(X) @ theta_ne) ** 2)
print(f"optimal training MSE = {mse_opt:.4f}")

### 1.2 Gradient descent
Write `gradient_descent(X, y, lr, n_iter)` for the loss $L(\theta)=\frac1n\lVert y-A\theta\rVert^2$ where `A = add_ones(X)`. Start from $\theta=0$; the gradient is $-\frac2n A^\top(y-A\theta)$. Return the final $\theta$ and the list of losses (one per iteration, computed **before** each update).

Then run it
* on the **raw** features with `lr = 3e-5` for 3000 iterations → `theta_raw, hist_raw`;
* on **standardised** features (`Xz = (X - X.mean(0)) / X.std(0)`) with `lr = 0.1` for 3000 iterations → `theta_z, hist_z`.

Plot `loss - mse_opt` for both runs on a log scale.

In [ ]:
# TODO
def gradient_descent(X, y, lr, n_iter):
    raise NotImplementedError("TODO")

mu, sd = X.mean(axis=0), X.std(axis=0)
Xz = (X - mu) / sd
theta_raw, hist_raw = None, None
theta_z, hist_z = None, None

In [ ]:
# CHECK (run this cell, it must pass)
assert len(hist_raw) == 3000 and len(hist_z) == 3000
assert hist_z[-1] - mse_opt < 1e-8, "standardised GD should have converged"
assert hist_raw[-1] - mse_opt > 1e-2, "raw GD should still be far from the optimum"
print("✅ gradient descent behaves as expected")

In [ ]:
# What happens with a slightly larger learning rate on the raw features?
_, h = gradient_descent(X, y, lr=1e-4, n_iter=30)
print("losses with lr=1e-4 on raw features:", np.round(h[:6], 1), "...", f"{h[-1]:.3e}")
A_raw, A_z = add_ones(X), add_ones(Xz)
print(f"condition number of A^T A: raw {np.linalg.cond(A_raw.T @ A_raw):.1e}, standardised {np.linalg.cond(A_z.T @ A_z):.1f}")

### 1.3 Back to the original units
`theta_z` describes the model in standardised units: $\hat y=\tilde b+\sum_j \tilde w_j\,(x_j-\mu_j)/s_j$. Convert it to $(b, w)$ in the original units → `theta_back`.

In [ ]:
# TODO
theta_back = None

In [ ]:
# CHECK (run this cell, it must pass)
assert np.allclose(theta_back, theta_ne, atol=1e-6)
print("✅ standardised GD recovers the exact solution")

### ✍️ Question 1
Explain, using the loss curves, the diverging run and the condition numbers, why gradient descent on the raw features is so slow, and why a bigger learning rate is not the fix. Which parameters had not converged in `theta_raw`?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 2 · Nucleus area from nucleus radius

**Context.** In the breast cancer dataset each tumour image gives the *mean radius* and *mean area* of its cell nuclei. Geometry suggests a relation, but not a linear one.

### 2.1 Data

In [ ]:
bc = load_breast_cancer(as_frame=True).frame
r = bc[["mean radius"]].to_numpy()
a = bc["mean area"].to_numpy()
r_train, r_test, a_train, a_test = train_test_split(r, a, test_size=0.25, random_state=0)
plt.scatter(r_train, a_train, s=8); plt.xlabel("mean radius"); plt.ylabel("mean area"); plt.show()

### 2.2 A straight line
Fit `LinearRegression` of area on radius on the training set → `lin`. Print the training $R^2$ (`r2_lin`) and plot the residuals against the radius.

In [ ]:
# TODO
lin = None
r2_lin = None

In [ ]:
# CHECK (run this cell, it must pass)
assert r2_lin > 0.95
print("✅ linear fit done")

### ✍️ Question 2
The $R^2$ is above 0.97. Is the straight line a good model? What does the residual plot show, and what feature would you add?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

### 2.3 Three candidate models
Compare by 5-fold cross-validated **RMSE** on the training set (`KFold(5, shuffle=True, random_state=0)`, scoring `"neg_root_mean_squared_error"`):
* **A**: `LinearRegression` on $r$;
* **B**: full quadratic, `PolynomialFeatures(2, include_bias=False)` then `LinearRegression`;
* **C**: area $= w\,r^2$, i.e. `FunctionTransformer(np.square)` then `LinearRegression(fit_intercept=False)`.

Store the mean RMSEs in a dict `cv_rmse` with keys `"A"`, `"B"`, `"C"`. Fit C on the whole training set (`model_c`) and print its coefficient.

In [ ]:
# TODO
kf = KFold(5, shuffle=True, random_state=0)
models = {}
cv_rmse = {}
model_c = None

In [ ]:
# CHECK (run this cell, it must pass)
assert set(cv_rmse) == {"A", "B", "C"}
assert cv_rmse["C"] < 0.5 * cv_rmse["A"] and cv_rmse["B"] < 0.5 * cv_rmse["A"]
assert 2.9 < model_c[-1].coef_[0] < 3.3
print("✅ model comparison done")

### 2.4 Test once
Report the test RMSE and $R^2$ of the model you choose (store them in `rmse_test`, `r2_test`).

In [ ]:
# TODO
rmse_test = r2_test = None

In [ ]:
# CHECK (run this cell, it must pass)
assert r2_test > 0.99
print("✅ test evaluation done")

### ✍️ Question 3
Which model do you choose between B and C, and why? Interpret the coefficient of C. Why is this still called *linear* regression?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

---
# Exercise 3 · Alcohol content of wines

**Context.** 178 Italian wines, 13 chemical measurements. We predict **alcohol** (% vol) from the 12 other measurements.

### 3.1 Data and correlated features

In [ ]:
wine = load_wine(as_frame=True).data
yw = wine["alcohol"]
Xw = wine.drop(columns="alcohol")
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.25, random_state=0)
print("train:", Xw_train.shape, "  test:", Xw_test.shape)

C = Xw_train.corr().abs()
pairs = C.where(np.triu(np.ones(C.shape, dtype=bool), 1)).stack().sort_values(ascending=False)
print("\nmost correlated feature pairs:"); print(pairs.head(4).round(3))

### 3.2 Four models
Compare by 5-fold CV $R^2$ on the training set (`kf` from Exercise 2): a `DummyRegressor()`, a scaled `LinearRegression`, a scaled `RidgeCV(alphas=np.logspace(-3, 3, 50))` and a scaled `KNeighborsRegressor(n_neighbors=10)`. Store a DataFrame `cv_table` with columns `model`, `mean_r2`, `std_r2`.

In [ ]:
# TODO
cv_table = None

In [ ]:
# CHECK (run this cell, it must pass)
assert list(cv_table.columns) == ["model", "mean_r2", "std_r2"] and len(cv_table) == 4
d = cv_table.set_index("model")["mean_r2"]
assert d["linear"] > d["dummy"] + 0.3
print("✅ model comparison done")

### 3.3 How stable are the coefficients?
Draw 200 bootstrap samples of the training set (`rng = np.random.default_rng(0)`, sample rows **with replacement**), fit the scaled linear regression on each, and store the coefficients in an array `boot` of shape `(200, 12)`. Draw a horizontal boxplot per feature and compute, for each feature, the fraction of bootstrap fits where the coefficient is positive (`frac_pos`, a Series indexed by feature name).

In [ ]:
# TODO
boot = None
frac_pos = None

In [ ]:
# CHECK (run this cell, it must pass)
assert boot.shape == (200, 12)
assert frac_pos.between(0, 1).all()
print("✅ bootstrap done")

### 3.4 Test once
Fit the model you choose on the whole training set and report its test $R^2$ (`r2_wine_test`) next to the dummy baseline's.

In [ ]:
# TODO
r2_wine_test = None

In [ ]:
# CHECK (run this cell, it must pass)
assert r2_wine_test > 0.3
print("✅ done")

### ✍️ Question 4
(a) Which model did you choose in 3.4, and why, given the CV table?
(b) Which coefficients can you interpret with some confidence, and which cannot? Relate your answer to the correlated pairs of 3.1.

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*